In [36]:
import pandas as pd

df = pd.read_csv('../dataset/email_spam_indo.csv')
df.head()

,Kategori,Pesan
0,spam,Secara alami tak tertahankan identitas perusah...
1,spam,Fanny Gunslinger Perdagangan Saham adalah Merr...
2,spam,Rumah -rumah baru yang luar biasa menjadi muda...
3,spam,4 Permintaan Khusus Pencetakan Warna Informasi...
4,spam,"Jangan punya uang, dapatkan CD perangkat lunak..."


In [37]:
df = df.rename(columns={
    'Pesan': 'text',
    'Kategori': 'label'
})

In [38]:
import re

def preprocess(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'(dikeret oleh|ditahan oleh|ect pada|subjek:).*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\b\w+\s+(com|net|org|co|id)\b', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'^(re|fw|fwd):[^.?!\n]*', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text'].apply(preprocess)
df[['text', 'clean_text']].head()

,text,clean_text
0,Secara alami tak tertahankan identitas perusah...,secara alami tak tertahankan identitas perusah...
1,Fanny Gunslinger Perdagangan Saham adalah Merr...,fanny gunslinger perdagangan saham adalah merr...
2,Rumah -rumah baru yang luar biasa menjadi muda...,rumah rumah baru yang luar biasa menjadi mudah...
3,4 Permintaan Khusus Pencetakan Warna Informasi...,permintaan khusus pencetakan warna informasi t...
4,"Jangan punya uang, dapatkan CD perangkat lunak...",jangan punya uang dapatkan cd perangkat lunak ...


In [39]:
NORMALIZATION_DICT = {
    # 🔤 Singkatan umum
    "gk": "tidak",
    "d": "di",
    "ga": "tidak",
    "gak": "tidak",
    "nggak": "tidak",
    "tdk": "tidak",
    "dr": "dari",
    "dgn": "dengan",
    "utk": "untuk",
    "yg": "yang",
    "jg": "juga",
    "krn": "karena",
    "blm": "belum",
    "sdh": "sudah",
    "aja": "saja",
    "trs": "terus",
    "bgt": "banget",
    "dlm": "dalam",

    # 💰 Kata khas spam / promo
    "rp": "rupiah",
    "jt": "juta",
    "rb": "ribu",
    "cashback": "cashback",
    "promo": "promosi",
    "diskon": "diskon",
    "gratis": "gratis",
    "hadiah": "hadiah",
    "menang": "menang",
    "undian": "undian",

    # 📱 Variasi kata penting
    "tlp": "telepon",
    "telp": "telepon",
    "hp": "handphone",
    "no": "nomor",
    "rek": "rekening",
    "an": "atas nama",

    # ⚡ Variasi penulisan spam
    "selamat!!!": "selamat",
    "selamat!!": "selamat",
    "selamat!": "selamat",
    "trnsfer": "transfer",
    "trf": "transfer",

    # 🧩 Variasi angka ke huruf (leetspeak)
    "4nda": "anda",
    "4pa": "apa",
    "1ni": "ini",
    "k4mu": "kamu",

    # 🪤 Kata clickbait spam
    "klik": "klik",
    "segera": "segera",
    "buruan": "cepat",
    "cepat": "cepat",
    "terbatas": "terbatas",
    "khusus": "khusus",

    # 🎯 Kata inti spam
    "free": "gratis",
    "win": "menang",
    "winner": "pemenang",
    "prize": "hadiah",
    "bonus": "bonus",
    "gift": "hadiah",
    "reward": "hadiah",

    # 💰 Finansial / uang
    "cash": "uang",
    "money": "uang",
    "credit": "kredit",
    "loan": "pinjaman",
    "bank": "bank",
    "transfer": "transfer",
    "account": "akun",
    "sspecial": "spesial",
    "balance": "saldo",

    # 📢 Promosi / marketing
    "promo": "promosi",
    "offer": "penawaran",
    "deal": "penawaran",
    "discount": "diskon",
    "sale": "diskon",
    "limited": "terbatas",
    "exclusive": "eksklusif",

    # ⚡ Call to action (penting banget buat spam)
    "click": "klik",
    "claim": "klaim",
    "buy": "beli",
    "order": "pesan",
    "register": "daftar",
    "subscribe": "langganan",
    "join": "gabung",
    "verify": "verifikasi",
    "confirm": "konfirmasi",

    # ⏰ urgency
    "urgent": "segera",
    "now": "sekarang",
    "today": "hari ini",
    "instant": "instan",

    # 🖥️ teknologi / produk
    "software": "perangkat lunak",
    "system": "sistem",
    "update": "pembaruan",
    "download": "unduh",
    "install": "instal",

    # 📧 email umum
    "hello": "halo",
    "dear": "halo",
    "sir": "bapak",
    "madam": "ibu",

    # 🧩 lain-lain yang sering muncul
    "service": "layanan",
    "customer": "pelanggan",
    "support": "dukungan",
    "information": "informasi",
    "message": "pesan"
}
def normalize_text(text):
    words = text.split()
    normalized_words = [NORMALIZATION_DICT.get(word, word) for word in words]
    return ' '.join(normalized_words)

df['normalized_text'] = df['clean_text'].apply(normalize_text)
df[['text', 'clean_text','normalized_text']].head()


,text,clean_text,normalized_text
0,Secara alami tak tertahankan identitas perusah...,secara alami tak tertahankan identitas perusah...,secara alami tak tertahankan identitas perusah...
1,Fanny Gunslinger Perdagangan Saham adalah Merr...,fanny gunslinger perdagangan saham adalah merr...,fanny gunslinger perdagangan saham adalah merr...
2,Rumah -rumah baru yang luar biasa menjadi muda...,rumah rumah baru yang luar biasa menjadi mudah...,rumah rumah baru yang luar biasa menjadi mudah...
3,4 Permintaan Khusus Pencetakan Warna Informasi...,permintaan khusus pencetakan warna informasi t...,permintaan khusus pencetakan warna informasi t...
4,"Jangan punya uang, dapatkan CD perangkat lunak...",jangan punya uang dapatkan cd perangkat lunak ...,jangan punya uang dapatkan cd perangkat lunak ...


In [40]:
!pip install Sastrawi
!pip install nltk


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [41]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# ambil stopword default
factory = StopWordRemoverFactory()
stopwords = set(factory.get_stop_words())

# fungsi hapus stopword
def remove_stopwords(text):
    return ' '.join([word for word in text.split() if word not in stopwords])

# apply ke data
df['stopwords'] = df['normalized_text'].apply(remove_stopwords)

# lihat hasil
df[['text', 'normalized_text', 'stopwords']].head()

,text,normalized_text,stopwords
0,Secara alami tak tertahankan identitas perusah...,secara alami tak tertahankan identitas perusah...,alami tak tertahankan identitas perusahaan san...
1,Fanny Gunslinger Perdagangan Saham adalah Merr...,fanny gunslinger perdagangan saham adalah merr...,fanny gunslinger perdagangan saham merrill muz...
2,Rumah -rumah baru yang luar biasa menjadi muda...,rumah rumah baru yang luar biasa menjadi mudah...,rumah rumah baru luar biasa menjadi mudah menu...
3,4 Permintaan Khusus Pencetakan Warna Informasi...,permintaan khusus pencetakan warna informasi t...,permintaan khusus pencetakan warna informasi t...
4,"Jangan punya uang, dapatkan CD perangkat lunak...",jangan punya uang dapatkan cd perangkat lunak ...,jangan punya uang dapatkan cd perangkat lunak ...


In [42]:
# tokenization
df['tokens'] = df['stopwords'].apply(lambda x: x.split())

# lihat hasil
df[['text', 'clean_text', 'stopwords', 'tokens']].head()

,text,clean_text,stopwords,tokens
0,Secara alami tak tertahankan identitas perusah...,secara alami tak tertahankan identitas perusah...,alami tak tertahankan identitas perusahaan san...,"[alami, tak, tertahankan, identitas, perusahaa..."
1,Fanny Gunslinger Perdagangan Saham adalah Merr...,fanny gunslinger perdagangan saham adalah merr...,fanny gunslinger perdagangan saham merrill muz...,"[fanny, gunslinger, perdagangan, saham, merril..."
2,Rumah -rumah baru yang luar biasa menjadi muda...,rumah rumah baru yang luar biasa menjadi mudah...,rumah rumah baru luar biasa menjadi mudah menu...,"[rumah, rumah, baru, luar, biasa, menjadi, mud..."
3,4 Permintaan Khusus Pencetakan Warna Informasi...,permintaan khusus pencetakan warna informasi t...,permintaan khusus pencetakan warna informasi t...,"[permintaan, khusus, pencetakan, warna, inform..."
4,"Jangan punya uang, dapatkan CD perangkat lunak...",jangan punya uang dapatkan cd perangkat lunak ...,jangan punya uang dapatkan cd perangkat lunak ...,"[jangan, punya, uang, dapatkan, cd, perangkat,..."


In [48]:
!pip install pycaret scikit-learn gensim


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [49]:
import pandas as pd
import numpy as np
from gensim.models import Word2Vec

# Model Word2Vec butuh data berformat list kata per kalimat (ada di df['tokens'])
# Kita buat 100 fitur/dimensi vektor untuk setiap kalimat
w2v_model = Word2Vec(sentences=df['tokens'], vector_size=100, window=5, min_count=2, workers=4)

# Fungsi pembuat Sentence/Document Vector (Rata-rata fitur vektor dari kumpulan kata)
def document_vector(doc):
    valid_words = [word for word in doc if word in w2v_model.wv.key_to_index]
    if len(valid_words) == 0:
        return np.zeros(100)
    return np.mean(w2v_model.wv[valid_words], axis=0)

X_w2v = np.array(df['tokens'].apply(document_vector).tolist())

# Susun ke bentuk DataFrame untuk PyCaret
feature_names = [f"w2v_{i}" for i in range(100)]
df_pycaret = pd.DataFrame(X_w2v, columns=feature_names)
df_pycaret['label'] = df['label'].values

df_pycaret.head()

,w2v_0,w2v_1,w2v_2,w2v_3,w2v_4,w2v_5,w2v_6,w2v_7,w2v_8,w2v_9,...,w2v_91,w2v_92,w2v_93,w2v_94,w2v_95,w2v_96,w2v_97,w2v_98,w2v_99,label
0,0.134489,0.807342,0.498257,-0.063721,-0.169234,-0.646569,0.154103,0.664968,-0.158069,-0.703296,...,0.048030,0.181323,0.371806,1.029468,0.115979,0.938906,-0.575445,0.490879,-0.352854,spam
1,0.032136,0.335430,0.295573,-0.014485,-0.060015,-0.420031,0.126776,0.472401,-0.232447,-0.380141,...,0.009139,0.128085,0.156844,0.597676,0.110688,0.377552,-0.457171,0.374415,-0.169355,spam
2,0.140246,0.621162,0.449726,-0.073455,-0.164225,-0.662005,0.120718,0.753601,-0.293653,-0.623552,...,0.039158,0.150984,0.449399,1.012958,0.142151,0.806985,-0.687587,0.520943,-0.355973,spam
3,0.097920,0.232096,0.085700,-0.222923,0.370899,-1.096216,-0.327943,1.596879,-0.112263,-0.021731,...,0.220650,0.647678,0.638499,1.069038,0.439811,0.245620,-0.396157,0.032345,-0.254146,spam
4,0.148030,0.589840,0.224422,0.014238,-0.155230,-0.333954,0.315105,0.892459,-0.695333,-0.173770,...,0.244014,0.248950,0.515625,1.199996,0.293515,0.496282,-0.203508,0.602058,-0.710671,spam


In [50]:
from pycaret.classification import ClassificationExperiment

# Inisialisasi Environment PyCaret berbasis Object Khusus untuk W2V
exp_w2v = ClassificationExperiment()
clf_setup = exp_w2v.setup(data=df_pycaret, target='label', session_id=42, use_gpu=False)

,Description,Value
0,Session id,42
1,Target,label
2,Target type,Binary
3,Target mapping,"ham: 0, spam: 1"
4,Original data shape,"(2636, 101)"
5,Transformed data shape,"(2636, 101)"
6,Transformed train set shape,"(1845, 101)"
7,Transformed test set shape,"(791, 101)"
8,Numeric features,100
9,Preprocess,True


In [51]:
# Membandingkan model SVM, Naive Bayes (nb), dan Logistic Regression (lr)
best_model = exp_w2v.compare_models(include=['svm', 'nb', 'lr'])

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
svm,SVM - Linear Kernel,0.9116,0.9694,0.9116,0.9146,0.9114,0.8230,0.8260,0.7530
lr,Logistic Regression,0.8921,0.9573,0.8921,0.8933,0.8919,0.7835,0.7850,0.0220
nb,Naive Bayes,0.7398,0.8718,0.7398,0.7705,0.7297,0.4722,0.5050,0.3380


In [58]:
# TES INPUT TEKS MAUPUN PERCAKAPAN
input_teks = "dapatkan promo ini sekarang"

# Mesin Penerjemah
teks_bersih = preprocess(input_teks)
teks_normal = normalize_text(teks_bersih)
teks_stopword = remove_stopwords(teks_normal)

# Ubah ke Vektor Word2Vec dan Matriks (Memanggil fungsi W2V yg di atas)
kata_kata = str(teks_stopword).split() 
vektor_teks = document_vector(kata_kata)
df_input = pd.DataFrame([vektor_teks], columns=feature_names)

# Tebak Kalimat!
prediksi = exp_w2v.predict_model(best_model, data=df_input)

print("\n" + "="*50)
print("Teks Anda   :", input_teks)
print("PREDIKSI NYA:", prediksi['prediction_label'].iloc[0].upper())
print("="*50)



Teks Anda   : dapatkan promo ini sekarang
PREDIKSI NYA: SPAM
